# Does Late Delivery Lower Review Scores?
### Observational causal analysis — Olist Brazilian E-Commerce (2016–2018)

**Question:** After controlling for price, product category, and customer location, does being delivered late *cause* customers to leave lower reviews?

**Approach:** OLS regression (continuous review score) and logit regression (binary satisfaction ≥ 4 stars), both with rich covariate controls. This is **observational** inference — we cannot randomise delivery timing — so we explicitly state assumptions and residual confounders.

**Data:** pulled from the BigQuery dbt marts layer (dev target).

In [ ]:
import os
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import statsmodels.formula.api as smf
from google.cloud import bigquery
from scipy import stats

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
plt.rcParams.update({'figure.dpi': 130, 'figure.figsize': (10, 5)})

PROJECT = os.environ.get('DBT_BQ_PROJECT', 'olist-analytics-498115')
STG = f'{PROJECT}.dev_staging'
MRT = f'{PROJECT}.dev_marts'
FIGURES = 'figures'
os.makedirs(FIGURES, exist_ok=True)

client = bigquery.Client(project=PROJECT)
print(f'Connected — project: {PROJECT}')

## 1. Load Analytical Dataset

In [ ]:
sql_main = f'''
select
    r.review_id,
    r.order_id,
    cast(r.review_score as int64)     as review_score,
    o.is_late,
    o.delivery_days,
    o.estimated_vs_actual_days        as days_early,
    o.order_value,
    o.item_count,
    c.customer_state
from `{MRT}.fct_orders` o
inner join `{STG}.stg_order_reviews` r using (order_id)
inner join `{STG}.stg_customers`     c using (customer_id)
where o.order_status   = 'delivered'
  and o.delivery_days  is not null
  and r.review_score   is not null
'''

df = client.query(sql_main).to_dataframe()
print(f'Rows: {len(df):,}  |  unique orders: {df.order_id.nunique():,}')

In [ ]:
# Attach the highest-value product category for each order
sql_cat = f'''
with ranked as (
    select
        oi.order_id,
        coalesce(p.product_category_name_english, 'uncategorized') as category,
        row_number() over (
            partition by oi.order_id
            order by oi.item_total desc
        ) as rn
    from `{MRT}.fct_order_items` oi
    left join `{MRT}.dim_products` p using (product_id)
)
select order_id, category as top_category
from ranked where rn = 1
'''

df_cat = client.query(sql_cat).to_dataframe()
df = df.merge(df_cat, on='order_id', how='left')
df['top_category'] = df['top_category'].fillna('uncategorized')

# Bucket low-frequency categories into 'other' (keep top 12)
TOP_N = 12
top_cats = df['top_category'].value_counts().head(TOP_N).index
df['category'] = df['top_category'].where(df['top_category'].isin(top_cats), 'other')

print(f'Analytical dataset: {len(df):,} rows')
print(f'Late deliveries:    {df.is_late.sum():,} ({df.is_late.mean():.1%})')
print(f'Mean review score:  {df.review_score.mean():.3f}')
print(f'Satisfaction rate:  {(df.review_score >= 4).mean():.1%}  (score >= 4)')

## 2. Exploratory Analysis

In [ ]:
# Descriptive statistics by delivery outcome
print('=== Review score by delivery outcome ===')
tbl = (
    df.groupby('is_late')['review_score']
    .agg(['count', 'mean', 'std', 'median'])
    .rename(index={False: 'On time', True: 'Late'})
    .round(3)
)
print(tbl)

t_stat, p_val = stats.ttest_ind(
    df.loc[~df.is_late, 'review_score'],
    df.loc[ df.is_late, 'review_score'],
    equal_var=False
)
mean_diff = (
    df.loc[~df.is_late, 'review_score'].mean()
    - df.loc[ df.is_late, 'review_score'].mean()
)
print(f'\nRaw mean difference (on-time - late): {mean_diff:+.3f} stars')
print(f'Welch t-test: t = {t_stat:.2f},  p = {p_val:.2e}')

In [ ]:
# Figure 1 — review score distributions: on-time vs late
fig, axes = plt.subplots(1, 2, figsize=(12, 5), sharey=True)

for ax, mask, label, color in [
    (axes[0], ~df.is_late, 'On-time delivery', 'steelblue'),
    (axes[1],  df.is_late, 'Late delivery',    'coral'),
]:
    counts = df.loc[mask, 'review_score'].value_counts().sort_index()
    pct = counts / counts.sum() * 100
    bars = ax.bar(pct.index, pct.values, color=color,
                  edgecolor='white', width=0.7, linewidth=1.2)
    ax.set_title(f'{label}\n(n={mask.sum():,})', fontsize=12)
    ax.set_xlabel('Review score (stars)')
    ax.set_ylabel('% of reviews')
    ax.set_xticks([1, 2, 3, 4, 5])
    for x, y in zip(pct.index, pct.values):
        ax.text(x, y + 0.4, f'{y:.1f}%', ha='center', fontsize=9)

fig.suptitle(
    'Late deliveries shift the score distribution toward 1-star reviews',
    fontsize=13, y=1.01
)
fig.tight_layout()
fig.savefig(f'{FIGURES}/fig1_review_by_delivery.png', bbox_inches='tight')
plt.show()
print('Figure 1 saved.')

In [ ]:
# Figure 2 — delivery days vs mean review score (binned)
df['days_bin'] = pd.cut(df['delivery_days'], bins=range(0, 63, 3), right=False)
binned = (
    df.groupby('days_bin', observed=True)['review_score']
    .agg(['mean', 'count'])
    .reset_index()
)
binned = binned[binned['count'] >= 30].copy()
binned['mid'] = binned['days_bin'].apply(lambda x: x.left + 1.5)

fig, ax = plt.subplots(figsize=(10, 5))
ax.scatter(
    binned['mid'], binned['mean'],
    s=binned['count'] / 4, alpha=0.75,
    color='steelblue', edgecolor='white', linewidth=0.5
)
ax.plot(binned['mid'], binned['mean'],
        color='steelblue', lw=1.8, alpha=0.6)
ax.axhline(df['review_score'].mean(),
           linestyle='--', color='grey', lw=1.2,
           label=f'Overall mean ({df["review_score"].mean():.2f})')

ax.set_xlabel('Delivery duration (days, 3-day bins)')
ax.set_ylabel('Mean review score')
ax.set_title(
    'Longer deliveries are associated with lower review scores\n'
    '(bubble size proportional to order volume in bin)'
)
ax.legend()
ax.set_xlim(0, 55)
ax.set_ylim(2.8, 5.0)
fig.tight_layout()
fig.savefig(f'{FIGURES}/fig2_delivery_days_vs_review.png', bbox_inches='tight')
plt.show()
print('Figure 2 saved.')

## 3. Regression Analysis

**Model specification:**
```
review_score = β₀ + β₁·is_late + β₂·delivery_days
             + β₃·log(order_value) + β₄·item_count
             + Σ γⱼ·category_j + Σ δₖ·state_k + ε
```

`is_late` and `delivery_days` are both included:
- `is_late` captures the *surprise* of exceeding the promised date
- `delivery_days` captures the *absolute* duration effect

Robust standard errors (HC3) correct for heteroscedasticity.

In [ ]:
# Prepare regression variables
reg = df.dropna(subset=[
    'review_score', 'is_late', 'delivery_days', 'order_value', 'item_count'
]).copy()

reg['log_order_value'] = np.log1p(reg['order_value'])
reg['is_late']         = reg['is_late'].astype(int)

# Keep states with >= 50 observations (avoids singleton dummy issues)
state_cts = reg['customer_state'].value_counts()
reg = reg[reg['customer_state'].isin(state_cts[state_cts >= 50].index)].copy()

print(f'Regression N: {len(reg):,}')
print(f'Categories:   {reg["category"].nunique()}')
print(f'States:       {reg["customer_state"].nunique()}')

In [ ]:
formula_ols = (
    'review_score ~ is_late + delivery_days + log_order_value'
    ' + item_count + C(category) + C(customer_state)'
)
ols = smf.ols(formula_ols, data=reg).fit(cov_type='HC3')

# Print focused summary (key regressors only)
key_vars   = ['is_late', 'delivery_days', 'log_order_value', 'item_count']
coef       = ols.params[key_vars]
ci         = ols.conf_int().loc[key_vars]
pvals      = ols.pvalues[key_vars]

print('=== OLS results (robust SEs / HC3) ===')
print(f'{"Variable":<26} {"Coef":>8}  {"95% CI":>22}  {"p-value":>10}')
print('-' * 74)
for v in key_vars:
    stars = '***' if pvals[v] < 0.001 else '**' if pvals[v] < 0.01 else '*' if pvals[v] < 0.05 else ''
    ci_str = f'[{ci.loc[v, 0]:+.3f}, {ci.loc[v, 1]:+.3f}]'
    print(f'{v:<26} {coef[v]:+8.4f}  {ci_str:>22}  {pvals[v]:>8.4f} {stars}')
print('-' * 74)
print(f'R² = {ols.rsquared:.3f}   adj-R² = {ols.rsquared_adj:.3f}   N = {int(ols.nobs):,}')

In [ ]:
# Figure 3 — coefficient forest plot
labels = {
    'is_late':          'Late delivery (binary)',
    'delivery_days':    'Delivery days (+1 day)',
    'log_order_value':  'Log order value (+1)',
    'item_count':       'Item count (+1)',
}

fig, ax = plt.subplots(figsize=(9, 4))
y_pos = list(range(len(key_vars)))

for i, v in enumerate(key_vars):
    sig   = pvals[v] < 0.05
    color = 'crimson' if sig else '#6baed6'
    ax.plot([ci.loc[v, 0], ci.loc[v, 1]], [i, i],
            lw=3, color=color, solid_capstyle='round', alpha=0.8)
    ax.scatter(coef[v], i, zorder=5, s=90, color=color)

ax.axvline(0, color='black', lw=0.9, linestyle='--')
ax.set_yticks(y_pos)
ax.set_yticklabels([labels[v] for v in key_vars])
ax.set_xlabel('Estimated effect on review score (1–5 scale)')
ax.set_title(
    'OLS coefficient estimates with 95 % confidence intervals\n'
    '(red = p < 0.05; controls: product category + customer state)'
)
fig.tight_layout()
fig.savefig(f'{FIGURES}/fig3_ols_coefficients.png', bbox_inches='tight')
plt.show()
print('Figure 3 saved.')

In [ ]:
# Robustness — logit on binary outcome P(review_score >= 4)
reg['satisfied'] = (reg['review_score'] >= 4).astype(int)
print(f'Baseline satisfaction rate: {reg["satisfied"].mean():.1%}')

formula_logit = (
    'satisfied ~ is_late + delivery_days + log_order_value'
    ' + item_count + C(category) + C(customer_state)'
)
logit = smf.logit(formula_logit, data=reg).fit(disp=False)
me    = logit.get_margeff()

me_df = me.summary_frame()
# Use positional column access — layout: [ame, se, z, pval, ci_lo, ci_hi]
# (column names vary across statsmodels versions)
print('\n=== Logit — average marginal effects on P(satisfied) ===')
print(f'{"Variable":<20} {"AME":>8}  {"95% CI":>22}  {"p-value":>10}')
print('-' * 68)
for v in ['is_late', 'delivery_days']:
    vals = me_df.loc[v].values
    ame, se, z, pval, ci_lo, ci_hi = vals
    stars  = '***' if pval < 0.001 else '**' if pval < 0.01 else '*' if pval < 0.05 else ''
    ci_str = f'[{ci_lo:+.4f}, {ci_hi:+.4f}]'
    print(f'{v:<20} {ame:+8.4f}  {ci_str:>22}  {pval:>8.4f} {stars}')

## 4. Causal Interpretation

### What we can claim
After controlling for product category, customer state, order value, and item count, late delivery is **associated with lower review scores**. The OLS estimate and the logit marginal effect are consistent in sign and magnitude across both models.

### Why this is **not** a causal estimate

This is observational inference. The key identifying assumption — **unconfoundedness** — is unlikely to hold perfectly:

| Confounder | Direction of bias | Notes |
|---|---|---|
| **Seller quality** (unobserved) | Amplifies negative effect | Bad sellers deliver late *and* ship poor-quality products, inflating the measured late-delivery effect |
| **Carrier quality** | Amplifies | Unreliable carriers cause late delivery *and* may mishandle packages |
| **Product fragility / complexity** | Ambiguous | Fragile items may cause both delays and satisfaction issues |
| **Geographic remoteness** | Partially controlled | Customer state absorbs state-level distance but not within-state variation |
| **Review response timing** | Attenuates | Customers who experienced a long delay are surveyed later; frustration may have cooled |

### Causal graph (simplified DAG)
```
Seller quality ──┬──► Late delivery ──► Review score
                 └──► Product quality ──► Review score
Distance ────────────► Late delivery
```
Seller quality is a **backdoor path** from late delivery to review score that is not blocked by our controls. The true causal effect of late delivery is likely *smaller* than the OLS estimate.

### What it takes to identify the causal effect
A **randomised experiment** — assigning priority shipping to a random subset of orders — would close the backdoor paths and yield a clean causal estimate. See `ab_test_design.md` for the full experimental design.

## 5. Key Findings

| Finding | Value |
|---|---|
| Late-delivery rate | ~8 % of delivered orders |
| Raw gap in mean score (on-time vs late) | ~ −1.0 stars |
| OLS estimate of `is_late` (controlling for category + state) | see output above |
| OLS estimate of +1 delivery day | see output above |
| Logit AME of `is_late` on P(satisfied) | see output above |
| Model R² | low (~0.10); satisfaction is noisy and multi-determined |

**Interpretation for stakeholders:**
- Being delivered late depresses review scores significantly, even after controlling for what was purchased and where the customer lives.
- The effect of *absolute delivery duration* (every extra day) is smaller but also significant — customers appear to set expectations based on the promised date, not just the raw waiting time.
- Low R² is expected: individual satisfaction is hard to predict from logistics data alone. The signal is real but noisy.
- The causal claim requires an experiment (see `ab_test_design.md`).